# Introduction to Artificial Intelligence - Homework Assignment 10 (5 pts.)
- Name:
- NETID:

**Attention to the Final Clue**
Due December 9, 2026.

This assignment covers:

- Scaled dot-product self-attention: queries, keys, values, and attention weights
- Masking padding, and backpropagating through softmax and attention
- Position information, residual connections, and a small encoder-only Transformer built from scratch
- Training the encoder, reading its attention patterns, and comparing it with PyTorch's encoder and the HW09 RNN
- Closing the case: decoding the note four ways and collecting the physical evidence

| Task | Description | Points |
|---|---|---:|
| 00 | Reopen the cipher evidence | 0 |
| 01 | Build self-attention; explain the scaling and the mask | 1.125 |
| 02 | Backpropagate through attention; explain the softmax gradient | 0.75 |
| 03 | Add position information and assemble the encoder | 1.125 |
| 04 | Train the encoder; interpret its attention | 0.50 |
| 05 | Compare with PyTorch's encoder and the HW09 RNN | 0.75 |
| 06 | Decode the note and collect the evidence | 0.25 |
| 07 | Report the results to Caulfield | 0.50 |
| 08 | Export the notebook | 0 |
| | **Total** | **5.00** |

Run the cells in order and answer the written questions. A few cells train small models and take a minute or two on Colab's CPU; each prints its progress as it goes.

### *Story Progression*

The trained RNN from HW09 did what the department asked: it turned **"v ybirq zheqrevat ze gurvfrasyblq …"** into plain English. The note is a confession to the murder of Mr. Theisenfloyd, and it ends with directions: a padlock combination and a locker on the second floor of Cushing.

Caulfield has read the decoded message a dozen times. He isn't ready to act on it yet. The RNN was the department's first sequence model; before anyone walks into Cushing with bolt cutters, he wants a second, independently built decoder to agree. If two very different machines read the note the same way, that reading is much harder to argue with in court.

So this week we build the architecture behind modern language models: the **Transformer**. Instead of reading one character at a time and carrying a memory forward, a Transformer lets every position look directly at every other position, and learns where to look. We'll build its core operation, **self-attention**, from scratch, train a small encoder on the same ROT13 pairs, and compare it with PyTorch's version and the frozen HW09 RNN. Then we'll go collect the evidence.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/story_second_decoder.png" class="img-responsive" alt="Evidence 1: Caulfield rereads the decoded confession while his partner sets up a second, independent decoder." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 1: Caulfield rereads the decoded confession while his partner sets up a second, independent decoder.</b></figcaption>
</div>

## Task 00: Reopen the Cipher Evidence

### Task 00-1: Description (0 pts.)

#### The Same Pipeline as HW08 and HW09

This cell supplies a compact, function-based version of the HW08 and HW09 pipeline, so a fresh Colab runtime has everything it needs:

- the **same 41-token vocabulary**, with the same token IDs as HW08's `Rot13Dataset`: `<PAD>`, `<EOS>`, `<UNK>`, `<SOS>`, the space, 26 lowercase letters, and 10 digits;
- `make_phrases`, which regenerates **the same phrases** as HW08's `Rot13Dataset` (its 2,000 training phrases, from NLTK's word list and the same seed), encoded with `codecs.encode(phrase, 'rot_13')`;
- `encode`, `decode`, `collate`, and `loader` (HW08's dataset methods as plain functions);
- `masked_loss`: HW09's `cross_entropy` and the logit gradient from its `backward`, averaged over real target tokens (including EOS), ignoring padding, and taking token IDs instead of one-hot targets;
- `evaluate_sequence` and `read_note`, which report **token accuracy** (fraction of non-padding characters correct) and **exact-sequence accuracy** (fraction of phrases with every character correct).

| File | Contents | When to use it |
|---|---|---|
| `cipher_note.txt` | The forensics lab's checked transcription of the note, still encoded | Task 06 |
| `rnn_reference.npz` | The weights of the scratch RNN trained in HW09 (embedding 32, hidden 64), frozen | Task 05 |

The note is never used for training. Nothing to implement here: run the cell and check the printed summary.

### Task 00-1: Code (0 pts.)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/wtheisen/nd-cse-30124-homeworks.git'

try:
    import google.colab
except ImportError:
    # Reuse a local checkout, including when launched from a homework subfolder.
    DATA_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / '.git').exists() and (p / 'evidence').is_dir()), None)
    if DATA_ROOT is None:
        DATA_ROOT = Path.cwd() / 'nd-cse-30124-homeworks'
else:
    # Leave the old checkout before deleting it, including on a setup-cell rerun.
    os.chdir('/content')
    DATA_ROOT = Path('/content/nd-cse-30124-homeworks')
    # Fetch fresh evidence every time this cell runs in Colab.
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)

if not DATA_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(DATA_ROOT)], check=True)

if not (DATA_ROOT / '.git').exists():
    raise RuntimeError(f'{DATA_ROOT} exists but is not a Git checkout. Choose a different clone location.')

# All later relative paths now start at the homework repository root.
os.chdir(DATA_ROOT)
DATA_ROOT = Path.cwd()

import copy
import json
import math
import random
import codecs
import nltk
import string
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

torch.set_num_threads(2)
torch.manual_seed(42)

EVIDENCE_DIR = DATA_ROOT / 'evidence' / 'homework10'
required = ['cipher_note.txt', 'rnn_reference.npz']
if any(not (EVIDENCE_DIR / name).is_file() for name in required):
    raise FileNotFoundError('HW10 evidence is missing. Rerun the setup cell with an internet connection, or check that your local repository contains the evidence/homework10 files.')
os.chdir(EVIDENCE_DIR)

letter_text = Path('cipher_note.txt').read_text().strip()

# Supplied from HW08: NLTK's word list and the same seeded phrases as HW08's Rot13Dataset.
nltk.download('words', quiet=True)
from nltk.corpus import words
word_list = [w.lower() for w in words.words() if w.isalpha()]

def make_phrases(num_phrases, seed):
    rng = random.Random(seed)   # the same seed always makes the same phrases
    phrases = []
    for _ in range(num_phrases):
        phrase_words = rng.choices(word_list, k=rng.randint(3, 12))
        phrase_words = [str(rng.randint(0, 99)) if rng.random() < 0.15 else word for word in phrase_words]
        phrases.append(' '.join(phrase_words))
    return phrases

phrases = make_phrases(2000, seed=1)   # HW08's 2,000 training phrases

# Supplied from HW08: vocabulary, tokenizer, padding, and loaders.
vocab = ('<PAD>', '<EOS>', '<UNK>', '<SOS>', ' ') + tuple(string.ascii_lowercase + string.digits)
char2idx = {ch: i for i, ch in enumerate(vocab)}
PAD, EOS, UNK = char2idx['<PAD>'], char2idx['<EOS>'], char2idx['<UNK>']
V = len(vocab)

def encode(text):
    return torch.tensor([char2idx.get(ch, UNK) for ch in text.lower()] + [EOS], dtype=torch.long)

def decode(tokens):
    text = []
    for index in tokens:
        index = int(index)
        if index == EOS:
            break
        if index == PAD:
            continue
        text.append(vocab[index] if len(vocab[index]) == 1 else '?')
    return ''.join(text)

def collate(batch):
    source, target = zip(*batch)
    return (pad_sequence(source, batch_first=True, padding_value=PAD),
            pad_sequence(target, batch_first=True, padding_value=PAD))

pairs = [(encode(codecs.encode(phrase, 'rot_13')), encode(phrase)) for phrase in phrases]

def loader(shuffle=False):
    return DataLoader(pairs, batch_size=64, shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(42))

# Supplied from HW09: stable log-softmax, then the mean cross-entropy over real (non-PAD) targets and its logit gradient.
def log_softmax(logits):
    shifted = logits - logits.max(dim=-1, keepdim=True).values
    return shifted - shifted.exp().sum(dim=-1, keepdim=True).log()

def target_log_probs(logits, targets):
    return log_softmax(logits).gather(-1, targets.unsqueeze(-1)).squeeze(-1)

def masked_mean_loss(logits, targets):
    valid = targets != PAD
    return -target_log_probs(logits, targets)[valid].sum() / valid.sum()

def masked_loss(logits, targets):
    valid = targets != PAD
    loss = masked_mean_loss(logits, targets)
    gradient = log_softmax(logits).exp() - F.one_hot(targets, logits.shape[-1]).to(logits.dtype)
    gradient = gradient * valid.unsqueeze(-1).to(logits.dtype) / valid.sum()
    return loss, gradient

# Supplied from HW08/HW09: padding-aware evaluation and note reading.
@torch.no_grad()
def evaluate_sequence(model):
    if isinstance(model, nn.Module):
        model.eval()
    loss_sum, correct, count, exact, examples = 0., 0, 0, 0, 0
    for x, y in loader():
        logits = model(x) if isinstance(model, nn.Module) else model.forward(x)
        valid = y != PAD
        pred = logits.argmax(dim=-1)
        loss_sum += float(F.cross_entropy(logits.transpose(1, 2), y, ignore_index=PAD, reduction='sum'))
        correct += int(((pred == y) & valid).sum())
        count += int(valid.sum())
        exact += int(((pred == y) | ~valid).all(dim=1).sum())
        examples += len(x)
    return {'loss': round(loss_sum / count, 5), 'token_accuracy': round(correct / count, 5),
            'exact_sequence_accuracy': round(exact / examples, 5)}

@torch.no_grad()
def read_note(model):
    if isinstance(model, nn.Module):
        model.eval()
    x = encode(letter_text).unsqueeze(0)
    logits = model(x) if isinstance(model, nn.Module) else model.forward(x)
    return decode(logits.argmax(dim=-1)[0])

print('Training phrases:', len(pairs))
print('Vocabulary size:', V)
print('Note length (characters):', len(letter_text))
print('Example pair:', codecs.encode(phrases[0], 'rot_13'), '->', phrases[0])

### Task 00-1: Expected Output (0 pts.)

```text
Training phrases: 2000
Vocabulary size: 41
Note length (characters): 262
Example pair: birerzcgl fgveevat nflyn ohgpurevat fureelzbbe -> overempty stirring asyla butchering sherrymoor
```

## Task 01: Build Self-Attention

### Task 01-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_01-1a_attention_context.png" alt="Self-attention lets every position look at every position at once, instead of passing information along an RNN hidden state. Query q_i = x_i W_Q^T, key k_j = x_j W_K^T, value v_j = x_j W_V^T." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_01-1b_scores_softmax_mask.png" alt="Scores s_ij = q_i dot k_j divided by sqrt(d); softmax over the keys gives attention weights that sum to 1 in each row. Padded keys get score minus infinity, so exactly zero weight. No causal mask is needed here." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_01-1c_worked_example.png" alt="Worked example with d = 2: q = [1, 0], keys [1, 0], [0, 1], [1, 1]. Dot products 1, 0, 1; scaled 0.7071, 0, 0.7071; softmax 0.4011, 0.1978, 0.4011; with key 2 masked 0.6698, 0.3302, 0." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `attention_weights(Q, K, pad_mask)`:

- **Inputs:** queries `Q` and keys `K`, each `(B, T, d)`, and a Boolean `pad_mask` of shape `(B, T)` that is `True` at padding positions.
- **Return:** attention weights `A` of shape `(B, T, T)`, where `A[b, i, j]` is how much query `i` uses key `j`.

1. Compute all scores at once with a batched matrix product, `Q @ K.transpose(-2, -1)`, divided by $\sqrt{d}$.
2. Set the scores of padded **keys** to `-inf` with `masked_fill`. The mask has shape `(B, T)`; index it as `pad_mask[:, None, :]` so it lines up with the key dimension of `(B, T, T)`.
3. Apply softmax over the last dimension (the keys).

#### Python Help

```python
scores = torch.zeros(1, 3, 3)
mask = torch.tensor([[False, False, True]])     # (B, T): position 2 is padding
mask[:, None, :].shape                          # torch.Size([1, 1, 3]): broadcasts over the query rows
scores.masked_fill(mask[:, None, :], float('-inf'))   # every row gets -inf in column 2
torch.softmax(torch.tensor([0., 0., float('-inf')]), dim=-1)   # tensor([0.5, 0.5, 0.0])
```

#### Useful Links

* [Vaswani et al. (2017), *Attention Is All You Need*](https://arxiv.org/abs/1706.03762), Section 3.2.1
* [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)
* [`Tensor.masked_fill`](https://pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)


### Task 01-1: Code (0.625 pts.)

In [ ]:
def attention_weights(Q, K, pad_mask):
    # TODO: Scaled dot-product scores, padded keys set to -inf, softmax over the keys.
    scores = None
    return None

Q_demo = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]])
K_demo = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]])
no_padding = torch.tensor([[False, False, False]])
key2_padding = torch.tensor([[False, False, True]])

A_demo = attention_weights(Q_demo, K_demo, no_padding)
print('Attention shape:', tuple(A_demo.shape))
print('Query 0, no padding:   ', A_demo[0, 0].numpy().round(4))
print('Query 0, key 2 masked: ', attention_weights(Q_demo, K_demo, key2_padding)[0, 0].numpy().round(4))
print('Every row sums to one:', torch.allclose(A_demo.sum(dim=-1), torch.ones(1, 3)))

### Task 01-1: Expected Output (0 pts.)

```text
Attention shape: (1, 3, 3)
Query 0, no padding:    [0.4011 0.1978 0.4011]
Query 0, key 2 masked:  [0.6698 0.3302 0.    ]
Every row sums to one: True
```

### Task 01-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_01-2a_attention_forward.png" alt="C = A V and Y = C W_O^T. Q, K, V = X W_Q^T, X W_K^T, X W_V^T, all (B, T, D); A = softmax(Q K^T / sqrt(D)) is (B, T, T); Y has the input shape (B, T, D), so blocks can be stacked." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_01-2b_worked_example.png" alt="Worked example: c_0 = 0.4011 [1, 0] + 0.1978 [0, 2] + 0.4011 [3, 1] = [1.6044, 0.7967]. One head, no biases; PyTorch masks use True for positions that may be attended to. attention_forward returns a cache for the backward pass." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `attention_forward(X, Wq, Wk, Wv, Wo, pad_mask)`:

1. Project `X` to `Q`, `K`, and `V`.
2. Compute `A` with your `attention_weights`.
3. Compute the context `C = A @ V` and the output `Y = C @ Wo.T`.

The supplied cache line and the comparison with PyTorch follow.

#### Python Help

```python
X = torch.zeros(2, 4, 6)          # (B, T, D)
W = torch.zeros(6, 6)             # (D_out, D_in), like HW06's LinearLayer
(X @ W.T).shape                   # torch.Size([2, 4, 6]): @ works on the last two dimensions
```

#### Useful Links

* [`F.scaled_dot_product_attention`](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)


### Task 01-2: Code (0.375 pts.)

In [ ]:
def attention_forward(X, Wq, Wk, Wv, Wo, pad_mask):
    # TODO: Project to queries, keys, and values; weight the values; apply the output projection.
    Q, K, Vals = None, None, None
    A = None
    C = None
    Y = None
    cache = (X, Q, K, Vals, A, C, Wq, Wk, Wv, Wo)
    return Y, cache

print('Worked example context:', (A_demo[0, 0] @ torch.tensor([[1., 0.], [0., 2.], [3., 1.]])).numpy().round(4))

# Supplied check against PyTorch on a padded batch, in double precision.
torch.manual_seed(10)
X_check = torch.randn(2, 4, 6, dtype=torch.float64)
W_check = [torch.randn(6, 6, dtype=torch.float64) / math.sqrt(6) for _ in range(4)]
pad_check = torch.tensor([[False, False, False, False], [False, False, True, True]])
Y_check, cache_check = attention_forward(X_check, *W_check, pad_check)
Q_c, K_c, V_c, A_c = cache_check[1], cache_check[2], cache_check[3], cache_check[4]
library = F.scaled_dot_product_attention(Q_c, K_c, V_c, attn_mask=(~pad_check)[:, None, :]) @ W_check[3].T
print('Output shape:', tuple(Y_check.shape), '| attention shape:', tuple(A_c.shape))
print('Matches F.scaled_dot_product_attention:', torch.allclose(Y_check, library, atol=1e-10))
print('Padded keys receive zero attention:', bool((A_c[1, :, 2:] == 0).all()))

### Task 01-2: Expected Output (0 pts.)

```text
Worked example context: [1.6044 0.7967]
Output shape: (2, 4, 6) | attention shape: (2, 4, 4)
Matches F.scaled_dot_product_attention: True
Padded keys receive zero attention: True
```

![Figure 1. One self-attention head. Every position is projected three ways; queries are compared with every key to produce a T × T table of weights, which mixes the values. Padded keys are masked to zero weight before the mix.](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/01_attention.png)

*Figure 1. One self-attention head. Every position is projected three ways; queries are compared with every key to produce a T × T table of weights, which mixes the values. Padded keys are masked to zero weight before the mix.*

### Task 01-3: Written Response (0.125 pts.)

1. Why do we divide the scores by $\sqrt{d}$? Describe what would happen to the attention weights, and to training, if the width were large and we didn't.
    * `[ANSWER]`
2. We mask padded **keys** (columns of `A`) but not padded **queries** (rows). Why is masking the keys essential, while leaving padded query rows unmasked is harmless?
    * `[ANSWER]`

## Task 02: Backpropagate Through Attention

### Task 02-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_02-1a_attention_backward.png" alt="Backward through attention: dW_O = (dL/dY)^T C, dC = (dL/dY) W_O; dA = dC V^T, dV = A^T dC; dS from the softmax rule; dQ = dS K / sqrt(d), dK = dS^T Q / sqrt(d); dW_Q = dQ^T X; dX = dQ W_Q + dK W_K + dV W_V, summing all three paths." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_02-1b_softmax_rule.png" alt="Softmax rule per row: dS_ij = A_ij (dA_ij minus the sum over k of dA_ik A_ik). Example: a = [0.7, 0.2, 0.1], dA = [1, 0, 0], weighted average 0.7, dS = [0.21, -0.14, -0.07], which sums to zero. Masked keys get zero gradient." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `attention_backward(dY, cache)`, following the table from top to bottom. The weight gradients are summed over every example **and** every position, because each weight matrix was used at every position; reshaping to `(B·T, D)` and doing one matrix product does that sum. The first and last gradients are supplied as examples.

Return the input gradient `dX` and a dictionary with keys `'Wq'`, `'Wk'`, `'Wv'`, `'Wo'`.

#### Python Help

```python
dY = torch.zeros(2, 4, 6)                 # (B, T, D)
dY.reshape(-1, 6).shape                   # torch.Size([8, 6]): every position of every example as one row
A = torch.zeros(2, 4, 4)
A.transpose(-2, -1)                       # swaps the last two dimensions, batch by batch
(A * A).sum(dim=-1, keepdim=True).shape   # torch.Size([2, 4, 1]): one number per row, ready to broadcast
```


### Task 02-1: Code (0.625 pts.)

In [ ]:
def attention_backward(dY, cache):
    X, Q, K, Vals, A, C, Wq, Wk, Wv, Wo = cache
    D = X.shape[-1]
    dWo = dY.reshape(-1, D).T @ C.reshape(-1, D)
    dC = dY @ Wo
    # TODO: Back through the weighted sum C = A @ Vals.
    dA = None
    dVals = None
    # TODO: Back through the row-wise softmax.
    dS = None
    # TODO: Back through the scaled scores S = Q @ K^T / sqrt(D).
    dQ = None
    dK = None
    # TODO: Weight gradients for the three projections, and the input gradient from all three paths.
    dWq = None
    dWk = None
    dWv = None
    dX = None
    return dX, {'Wq': dWq, 'Wk': dWk, 'Wv': dWv, 'Wo': dWo}

a_row = torch.tensor([[[0.7, 0.2, 0.1]]])
dA_row = torch.tensor([[[1., 0., 0.]]])
print('Worked example dS:', (a_row * (dA_row - (dA_row * a_row).sum(dim=-1, keepdim=True)))[0, 0].numpy().round(4))

# Supplied check against autograd on the same padded batch as Task 01-2.
torch.manual_seed(11)
upstream = torch.randn_like(Y_check)
dX_ours, grads_ours = attention_backward(upstream, cache_check)
X_auto = X_check.clone().requires_grad_()
W_auto = [w.clone().requires_grad_() for w in W_check]
Y_auto, _ = attention_forward(X_auto, *W_auto, pad_check)
(Y_auto * upstream).sum().backward()
print('dX matches autograd:', torch.allclose(dX_ours, X_auto.grad, atol=1e-10))
for name, w in zip(('Wq', 'Wk', 'Wv', 'Wo'), W_auto):
    print(f'd{name} matches autograd:', torch.allclose(grads_ours[name], w.grad, atol=1e-10))

### Task 02-1: Expected Output (0 pts.)

```text
Worked example dS: [ 0.21 -0.14 -0.07]
dX matches autograd: True
dWq matches autograd: True
dWk matches autograd: True
dWv matches autograd: True
dWo matches autograd: True
```

### Task 02-2: Written Response (0.125 pts.)

1. In the worked example, the three entries of $dS$ sum to zero. Explain why every row of $dS$ must sum to zero, using what happens to softmax when the same constant is added to every score.
    * `[ANSWER]`
2. Why does a padded key receive exactly zero gradient, and why is that what we want?
    * `[ANSWER]`

### *Story Progression*

The intern has been reading over Caulfield's shoulder again. "So the RNN reads the note like a person does, left to right, keeping a running summary. And this attention thing reads it like a detective with a corkboard: everything pinned up at once, with string between whatever's related." Caulfield allows that this is not a bad description.

"Then why," the intern asks, "would you ever use the RNN?" The answer turns out to be the same as in every other part of this case: it depends on the evidence. On a short, local cipher, both may work. The only way to know is to build the second one and check.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/story_two_readers.png" class="img-responsive" alt="Evidence 2: Two ways to read a note: one finger moving left to right, or everything pinned up at once with string between related clues." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 2: Two ways to read a note: one finger moving left to right, or everything pinned up at once with string between related clues.</b></figcaption>
</div>

## Task 03: Assemble the Encoder

### Task 03-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_03-1a_positional_encoding.png" alt="Attention treats its input as a set, so a positional encoding is added to each embedding: PE_t,2k = sin(t omega_k), PE_t,2k+1 = cos(t omega_k), omega_k = 10000 to the power -2k/D. We add 0.1 times PE." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_03-1b_clock_hands.png" alt="Early channel pairs oscillate quickly and later ones slowly, giving every position a unique fingerprint. Worked example with D = 4: PE_1 = [sin 1, cos 1, sin 0.01, cos 0.01] = [0.8415, 0.5403, 0.0100, 1.0000]; position 0 is [0, 1, 0, 1]." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `positional_encoding(length, width)`: return a `(length, width)` tensor with sines in the even channels and cosines in the odd channels. The positions and frequencies are supplied.

#### Python Help

```python
positions = torch.arange(3.)[:, None]       # shape (3, 1): one row per position
freqs = torch.tensor([1.0, 0.01])           # shape (2,)
(positions * freqs).shape                   # torch.Size([3, 2]): broadcasting makes a table
out = torch.zeros(3, 4)
out[:, 0::2] = 1.0                          # channels 0, 2  (start at 0, step 2)
out[:, 1::2] = 2.0                          # channels 1, 3  (start at 1, step 2)
```

#### Useful Links

* [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762), Section 3.5


### Task 03-1: Code (0.375 pts.)

In [ ]:
def positional_encoding(length, width):
    positions = torch.arange(length, dtype=torch.float32)[:, None]
    frequencies = torch.exp(torch.arange(0, width, 2, dtype=torch.float32) * (-math.log(10000.0) / width))
    result = torch.zeros(length, width)
    # TODO: Sines in the even channels, cosines in the odd channels.
    return result

print('PE for positions 0 and 1 (width 4):')
print(positional_encoding(2, 4).numpy().round(4))

# Supplied demonstration: attention alone can't tell order; positions can.
torch.manual_seed(3)
W_demo = [torch.randn(8, 8) / math.sqrt(8) for _ in range(4)]
X_seq = torch.randn(1, 5, 8)
order = torch.tensor([4, 3, 2, 1, 0])
no_pad = torch.zeros(1, 5, dtype=torch.bool)
Y_seq, _ = attention_forward(X_seq, *W_demo, no_pad)
Y_rev, _ = attention_forward(X_seq[:, order], *W_demo, no_pad)
print('Without positions, reversing the input just reverses the output:', torch.allclose(Y_rev, Y_seq[:, order], atol=1e-6))
pe = positional_encoding(5, 8)
Y_seq_pe, _ = attention_forward(X_seq + pe, *W_demo, no_pad)
Y_rev_pe, _ = attention_forward(X_seq[:, order] + pe, *W_demo, no_pad)
print('With positions added, it does not:', torch.allclose(Y_rev_pe, Y_seq_pe[:, order], atol=1e-6))

### Task 03-1: Expected Output (0 pts.)

```text
PE for positions 0 and 1 (width 4):
[[0.     1.     0.     1.    ]
 [0.8415 0.5403 0.01   1.    ]]
Without positions, reversing the input just reverses the output: True
With positions added, it does not: False
```

### Task 03-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_03-2a_encoder_block.png" alt="Encoder block: U = X + Attention(X), X_next = U + FFN(U), FFN(U) = ReLU(U W_1^T + b_1) W_2^T + b_2. Residuals pass gradient through because d(X + f(X)) = 1 + f prime." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>
<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_03-2b_our_encoder.png" alt="Our encoder: tokens, embedding plus 0.1 PE, two blocks, linear output. Parameters: E 984, per block attention 2,304 and FFN 2,376, output 1,025, total 11,369. Backward through a residual adds the identity path to the sublayer path." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete the marked lines of `ScratchEncoder`:

1. In `forward`: the two residual additions and the FFN between them (`U`, `pre`, `hidden`, and the new `X`).
2. In `backward`: the two places where the identity path's gradient is added back in.

The supplied check compares **every** parameter gradient of a tiny double-precision encoder with autograd, and confirms that adding extra padding doesn't change the outputs at real positions.

#### Python Help

```python
pre = torch.tensor([[-1.0, 2.0]])
pre.clamp_min(0)                  # ReLU: tensor([[0., 2.]])
(pre > 0)                         # tensor([[False,  True]]): ReLU's backward gate (HW07)
```


### Task 03-2: Code (0.75 pts.)

In [ ]:
class ScratchEncoder:
    def __init__(self, vocab_size=V, width=24, blocks=2):
        self.width, self.blocks = width, blocks
        self.params = {'E': torch.randn(vocab_size, width) * 0.1,
                       'Wout': torch.randn(vocab_size, width) / math.sqrt(width),
                       'bout': torch.zeros(vocab_size)}
        self.params['E'][PAD] = 0
        for i in range(blocks):
            for name in ('Wq', 'Wk', 'Wv', 'Wo'):
                self.params[f'{i}.{name}'] = torch.randn(width, width) / math.sqrt(width) * 0.5
            self.params[f'{i}.W1'] = torch.randn(2 * width, width) / math.sqrt(width) * 0.5
            self.params[f'{i}.b1'] = torch.zeros(2 * width)
            self.params[f'{i}.W2'] = torch.randn(width, 2 * width) / math.sqrt(2 * width) * 0.5
            self.params[f'{i}.b2'] = torch.zeros(width)

    def forward(self, tokens):
        p = self.params
        X = p['E'][tokens] + 0.1 * positional_encoding(tokens.shape[1], self.width).to(p['E'].dtype)
        pad_mask = tokens == PAD
        self.caches = []
        for i in range(self.blocks):
            b = f'{i}.'
            context, attn_cache = attention_forward(X, p[b + 'Wq'], p[b + 'Wk'], p[b + 'Wv'], p[b + 'Wo'], pad_mask)
            # TODO: Attention residual, then the position-wise FFN with its own residual.
            U = None
            pre = None
            hidden = None
            X = None
            self.caches.append((attn_cache, U, pre, hidden))
        self.tokens, self.final = tokens, X
        return X @ p['Wout'].T + p['bout']

    def backward(self, dlogits):
        p, D = self.params, self.width
        g = {'Wout': dlogits.reshape(-1, dlogits.shape[-1]).T @ self.final.reshape(-1, D),
             'bout': dlogits.sum(dim=(0, 1))}
        dX = dlogits @ p['Wout']
        for i in reversed(range(self.blocks)):
            b = f'{i}.'
            attn_cache, U, pre, hidden = self.caches[i]
            g[b + 'W2'] = dX.reshape(-1, D).T @ hidden.reshape(-1, 2 * D)
            g[b + 'b2'] = dX.sum(dim=(0, 1))
            dpre = (dX @ p[b + 'W2']) * (pre > 0)
            g[b + 'W1'] = dpre.reshape(-1, 2 * D).T @ U.reshape(-1, D)
            g[b + 'b1'] = dpre.sum(dim=(0, 1))
            # TODO: dU = identity path (dX) + the path back through the FFN.
            dU = None
            d_attention_input, attn_grads = attention_backward(dU, attn_cache)
            # TODO: dX for the block's input = identity path (dU) + the path back through attention.
            dX = None
            for name, value in attn_grads.items():
                g[b + name] = value
        g['E'] = torch.zeros_like(p['E'])
        g['E'].index_add_(0, self.tokens.reshape(-1), dX.reshape(-1, D))
        g['E'][PAD] = 0
        self.grads = g
        return dX

    def update(self, lr, max_norm=1.0):
        norm = torch.sqrt(sum((grad ** 2).sum() for grad in self.grads.values()))
        scale = min(1.0, max_norm / (float(norm) + 1e-12))
        for name in self.params:
            self.params[name] -= lr * scale * self.grads[name]
        self.params['E'][PAD] = 0
        return float(norm)

print('Parameters (width 24, 2 blocks):', sum(v.numel() for v in ScratchEncoder().params.values()))

# Supplied check: every gradient of a tiny double-precision encoder against autograd.
torch.manual_seed(5)
small = ScratchEncoder(V, width=4, blocks=2)
small.params = {k: v.double() for k, v in small.params.items()}
tiny_x, tiny_y = collate([(encode('nn'), encode('aa')), (encode('a'), encode('n'))])
tiny_logits = small.forward(tiny_x)
tiny_loss, tiny_dlogits = masked_loss(tiny_logits, tiny_y)
small.backward(tiny_dlogits)
auto = ScratchEncoder(V, width=4, blocks=2)
auto.params = {k: v.clone().requires_grad_() for k, v in small.params.items()}
F.cross_entropy(auto.forward(tiny_x).transpose(1, 2), tiny_y, ignore_index=PAD).backward()
mismatched = []
for name in small.params:
    expected = auto.params[name].grad.clone()
    expected[PAD] = 0 if name == 'E' else expected[PAD]
    if not torch.allclose(small.grads[name], expected, atol=1e-10):
        mismatched.append(name)
print(f'Gradients checked: {len(small.params)} parameter tensors | mismatches: {mismatched if mismatched else "none"}')
padded_x = torch.cat([tiny_x, torch.full((2, 3), PAD)], dim=1)
print('Extra padding leaves real-position outputs unchanged:',
      torch.allclose(small.forward(padded_x)[:, :tiny_x.shape[1]], tiny_logits, atol=1e-10))

### Task 03-2: Expected Output (0 pts.)

```text
Parameters (width 24, 2 blocks): 11369
Gradients checked: 19 parameter tensors | mismatches: none
Extra padding leaves real-position outputs unchanged: True
```

## Task 04: Train the Encoder

### Task 04-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_04-1_training_the_encoder.png" alt="Training loop: forward, masked loss, backward, clipped SGD update. 25 epochs at learning rate 0.4, printing the mean training loss. Loss starts near log 41 = 3.71 and should fall well below 1 within five epochs. The attention plot shows each block on its own color scale." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete the four lines inside the batch loop of `fit_scratch`. The loss tracking and printing are supplied.

#### Python Help

```python
loss, dlogits = masked_loss(logits, targets)   # HW09: returns the loss AND the logit gradient
model.backward(dlogits)
model.update(lr)                               # clips, then steps
```


### Task 04-1: Code (0.375 pts.)

In [ ]:
def fit_scratch(model, epochs, lr):
    history = []
    batches = loader(shuffle=True)
    for epoch in range(1, epochs + 1):
        loss_sum = 0.
        for x, y in batches:
            # TODO: Forward, masked loss, backward, clipped update.
            pass
            loss_sum += loss.item()
        history.append({'epoch': epoch, 'loss': loss_sum / len(batches)})
        if epoch == 1 or epoch % 5 == 0:
            print(f"Encoder epoch {epoch:2d}: mean training loss {loss_sum / len(batches):.4f}")
    return model, history

torch.manual_seed(42)
started = time.perf_counter()
encoder, encoder_history = fit_scratch(ScratchEncoder(), epochs=25, lr=0.4)
print(f'Training time: {time.perf_counter() - started:.0f} s')

# Supplied: plot both blocks' attention for one phrase.
def show_attention(model, text):
    tokens = encode(text).unsqueeze(0)
    model.forward(tokens)
    labels = list(text) + ['EOS']
    fig, axes = plt.subplots(1, model.blocks, figsize=(7 * model.blocks, 6.2))
    for block, ax in enumerate(axes):
        weights = model.caches[block][0][4][0].numpy()
        image = ax.imshow(weights, cmap='Blues', vmin=0)
        fig.colorbar(image, ax=ax, fraction=0.046)
        print(f'Block {block}: largest weight {weights.max():.3f}, smallest {weights.min():.3f} (uniform would be 1/{len(labels)} = {1 / len(labels):.3f})')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_xlabel('input position (key)')
        ax.set_ylabel('output position (query)')
        ax.set_title(f'Block {block} attention')
    fig.suptitle(f'Where each position looks: "{text}"')
    fig.tight_layout()
    plt.show()

sample_phrase = '88 vulcanizate cacuminate drip'
sample = codecs.encode(sample_phrase, 'rot_13')
print('Sample phrase:', sample, '->', sample_phrase)
show_attention(encoder, sample)

### Task 04-1: Reference Output (0 pts.)

Training time depends on your machine; the other lines should match when you run the cells in order.

```text
Encoder epoch  1: mean training loss 2.4488
Encoder epoch  5: mean training loss 0.0581
Encoder epoch 10: mean training loss 0.0059
Encoder epoch 15: mean training loss 0.0018
Encoder epoch 20: mean training loss 0.0009
Encoder epoch 25: mean training loss 0.0006
Sample phrase: 88 ihypnavmngr pnphzvangr qevc -> 88 vulcanizate cacuminate drip
Block 0: largest weight 0.035, smallest 0.030 (uniform would be 1/31 = 0.032)
Block 1: largest weight 0.143, smallest 0.001 (uniform would be 1/31 = 0.032)
```

![Attention patterns of the trained encoder](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/reference_attention.png)

*Attention patterns of the trained encoder for one phrase. Compare the plot produced by your code with this reference image.*

### Task 04-2: Written Response (0.125 pts.)

Describe the pattern in your attention plots. Does each position mostly attend to itself, to its neighbors, or broadly across the phrase? Explain why that pattern makes sense for ROT13, a cipher that replaces each character independently of its neighbors.

* `[ANSWER]`

## Task 05: Compare the Decoders

### Task 05-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_05-1_pytorch_encoder.png" alt="nn.TransformerEncoderLayer adds multiple heads, layer normalization, biases, FFN width 48 with ReLU, dropout 0. Use batch_first=True and src_key_padding_mask = (tokens == PAD). Trained with Adam at 0.003 for 15 epochs. Not a controlled comparison." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

Complete `TorchEncoder.__init__`: the embedding, one encoder layer with the settings in the table (width 24, 2 heads, feed-forward width 48, no dropout, `batch_first=True`), a two-layer `nn.TransformerEncoder`, and the output layer. `forward` is supplied. The supplied code trains it.

#### Python Help

```python
layer = nn.TransformerEncoderLayer(d_model=24, nhead=2, dim_feedforward=48, dropout=0.0, batch_first=True)
stack = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
```

#### Useful Links

* [`nn.TransformerEncoderLayer`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html) · [`nn.TransformerEncoder`](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html)
* [Layer normalization (Ba et al., 2016)](https://arxiv.org/abs/1607.06450)


### Task 05-1: Code (0.375 pts.)

In [ ]:
class TorchEncoder(nn.Module):
    def __init__(self, width=24):
        super().__init__()
        self.width = width
        # TODO: Embedding (padding row fixed at zero), two encoder layers, and the output layer.
        self.embed = None
        self.encoder = None
        self.output = None

    def forward(self, tokens):
        x = self.embed(tokens) + 0.1 * positional_encoding(tokens.shape[1], self.width)
        return self.output(self.encoder(x, src_key_padding_mask=(tokens == PAD)))

def fit_torch(model, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    batches = loader(shuffle=True)
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.
        for x, y in batches:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x).transpose(1, 2), y, ignore_index=PAD)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item()
        history.append({'epoch': epoch, 'loss': loss_sum / len(batches)})
        if epoch == 1 or epoch % 5 == 0:
            print(f"PyTorch epoch {epoch:2d}: mean training loss {loss_sum / len(batches):.4f}")
    return model, history

torch.manual_seed(42)
torch_encoder = TorchEncoder()
print('PyTorch encoder parameters:', sum(p.numel() for p in torch_encoder.parameters()))
started = time.perf_counter()
torch_encoder, torch_history = fit_torch(torch_encoder, epochs=15, lr=0.003)
print(f'Training time: {time.perf_counter() - started:.0f} s')

### Task 05-1: Reference Output (0 pts.)

```text
PyTorch encoder parameters: 11753
PyTorch epoch  1: mean training loss 1.9320
PyTorch epoch  5: mean training loss 0.0288
PyTorch epoch 10: mean training loss 0.0069
PyTorch epoch 15: mean training loss 0.0032
```

### Task 05-2: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_05-2_three_decoders.png" alt="Three decoders scored on the training phrases: the frozen HW09 RNN, the scratch encoder and the PyTorch encoder, with the direct ROT13 rule as a baseline. Exact-sequence accuracy counts a phrase wrong if any character is wrong. Reading new text is tested by the note in Task 06." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

#### Your Task

For each `(name, model)` pair in `decoders`, evaluate it on the training phrases with `evaluate_sequence` and append a dictionary with its name and metrics to `results`. The supplied code prints the table and plots the two encoders' training-loss histories.


### Task 05-2: Code (0.25 pts.)

In [ ]:
# Supplied: HW08's network classes (forward pass only), used to read the frozen HW09 checkpoint.
class LinearLayer:
    def __init__(self, input_dim, output_dim):
        self.W = torch.zeros(output_dim, input_dim)
        self.b = torch.zeros(output_dim)

    def forward(self, X):
        return X @ self.W.T + self.b

class EmbeddingLayer(LinearLayer):
    def __init__(self, vocab_size, embed_dim):
        super().__init__(input_dim=vocab_size, output_dim=embed_dim)
        self.vocab_size = vocab_size

    def forward(self, X_idx):
        return F.one_hot(X_idx, num_classes=self.vocab_size).float() @ self.W.T

class RecurrentBlock:
    def __init__(self, input_dim, hidden_size):
        self.Wx = torch.zeros(hidden_size, input_dim)
        self.Wc = torch.zeros(hidden_size, hidden_size)
        self.b = torch.zeros(hidden_size)

    def forward(self, X):
        c = torch.zeros(X.shape[0], self.Wc.shape[0])
        outputs = []
        for timestep in range(X.shape[1]):
            c = torch.tanh(X[:, timestep, :] @ self.Wx.T + c @ self.Wc.T + self.b)
            outputs.append(c)
        return torch.stack(outputs, dim=1)

class RecurrentNeuralNetwork:
    def __init__(self, vocab_size, embed_size, hidden_size):
        self.layers = [EmbeddingLayer(vocab_size, embed_size), RecurrentBlock(embed_size, hidden_size), LinearLayer(hidden_size, vocab_size)]

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

frozen_rnn = RecurrentNeuralNetwork(V, embed_size=32, hidden_size=64)
embedding_layer, recurrent_block, output_layer = frozen_rnn.layers
with np.load('rnn_reference.npz') as checkpoint:
    embedding_layer.W = torch.from_numpy(checkpoint['embedding_W'].copy())
    recurrent_block.Wx = torch.from_numpy(checkpoint['recurrent_W'].copy())
    recurrent_block.Wc = torch.from_numpy(checkpoint['recurrent_U'].copy())
    recurrent_block.b = torch.from_numpy(checkpoint['recurrent_b'].copy())
    output_layer.W = torch.from_numpy(checkpoint['output_W'].copy())
    output_layer.b = torch.from_numpy(checkpoint['output_b'].copy())
used = [embedding_layer.W, recurrent_block.Wx, recurrent_block.Wc, recurrent_block.b, output_layer.W, output_layer.b]
print('Frozen RNN parameters:', sum(p.numel() for p in used))

decoders = [('HW09 frozen RNN', frozen_rnn), ('Scratch encoder', encoder), ('PyTorch encoder', torch_encoder)]
results = []
for name, decoder in decoders:
    # TODO: Evaluate each decoder on the training phrases.
    pass

print('\nOn the 2,000 training phrases:')
print(f"{'model':<18}{'loss':>9}{'token acc':>11}{'exact seq':>11}")
for row in results:
    print(f"{row['model']:<18}{row['loss']:>9.5f}{row['token_accuracy']:>11.5f}{row['exact_sequence_accuracy']:>11.5f}")
print(f"{'Direct ROT13 rule':<18}{'—':>9}{1.0:>11.5f}{1.0:>11.5f}")

# Supplied: the two encoders' training-loss curves.
fig, ax = plt.subplots(figsize=(7, 4))
for name, history in [('Scratch encoder (SGD)', encoder_history), ('PyTorch encoder (Adam)', torch_history)]:
    ax.plot([row['epoch'] for row in history], [row['loss'] for row in history], label=name)
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('mean training loss (log scale)')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


### Task 05-2: Reference Output (0 pts.)

```text
Frozen RNN parameters: 10185

On the 2,000 training phrases:
model                  loss  token acc  exact seq
HW09 frozen RNN     0.00338    0.99961    0.97400
Scratch encoder     0.00057    1.00000    1.00000
PyTorch encoder     0.00300    1.00000    1.00000
Direct ROT13 rule         —    1.00000    1.00000
```

![Training-loss histories of the two encoders](https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/reference_encoder_histories.png)

*Training-loss histories of the two encoders. Compare the plot produced by your code with this reference image.*

### Task 05-3: Written Response (0.125 pts.)

1. Using your table, compare how well the RNN and the two encoders fit the training phrases. Every model in the table trained on these phrases. What does the table show, and why can't it tell Caulfield which decoder will read the note correctly?
    * `[ANSWER]`
2. Name **two** differences between the RNN and a Transformer encoder in how they process a sequence (think about order, memory, and what can run in parallel). Then explain why this ROT13 experiment cannot show that one architecture is better in general.
    * `[ANSWER]`

### *Story Progression*

Three machines, built three different ways, read the note. Caulfield lines up the outputs on his desk with the direct ROT13 decoding next to them. He isn't looking at accuracy tables anymore. He wants to know one thing: do they all say the same thing about where to go next?

They do. The locker number and the padlock combination are the same in every reading. Caulfield puts on his coat. The intern, for once, doesn't need to be asked to bring the flashlight.

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/story_locker.png" class="img-responsive" alt="Evidence 3: Following the decoded directions to locker 69." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 3: Following the decoded directions to locker 69.</b></figcaption>
</div>

## Task 06: Decode the Note and Collect the Evidence

### Task 06-1: Description (0 pts.)

<img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/plates/plate_06-1_rot13_rule.png" alt="ROT13 shifts each letter 13 places, wrapping from z to a, and is its own inverse. g = 6 becomes 19 = t; u = 20 becomes 7 = h; r = 17 becomes 4 = e, so gur decodes to the. Digits are unchanged." style="display: block; max-width: 900px; width: 100%; height: auto; margin: 0 auto 12px;"/>

The supplied cell applies the rule to the note, then reads the note with all three learned decoders and checks whether each reading agrees with the rule exactly. There's no code to write: the question is whether three machines built three different ways tell Caulfield the same thing.


### Task 06-1: Code (0 pts.)

In [ ]:
def rot13(text):
    # Supplied: shift each lowercase letter 13 places (wrapping around); leave everything else unchanged.
    letters = string.ascii_lowercase
    decoded = []
    for ch in text:
        if ch in letters:
            decoded.append(letters[(letters.index(ch) + 13) % 26])
        else:
            decoded.append(ch)
    return ''.join(decoded)

print('rot13("gur") =', rot13('gur'), '| applied twice:', rot13(rot13('gur')))
verified_message = rot13(letter_text)
print('Rule undoes codecs on every training phrase:', all(rot13(codecs.encode(phrase, 'rot_13')) == phrase for phrase in phrases))
print()
print('Decoded note (direct rule):')
print(verified_message)
print()
for name, decoder in decoders:
    reading = read_note(decoder)
    wrong = sum(a != b for a, b in zip(reading, verified_message)) + abs(len(reading) - len(verified_message))
    print(f'{name:<16} agrees with the rule: {reading == verified_message} ({wrong} characters differ)')

### Task 06-1: Reference Output (0 pts.)

```text
rot13("gur") = the | applied twice: gur
Rule undoes codecs on every training phrase: True

Decoded note (direct rule):
i loved murdering mr theisenfloyd at the estate the gas was the perfect murder weapon im sure ill get away with it as well hopefully nobody figures out the combination of the padlock play on locker 69 on the second floor of cushing otherwise i am in real trouble

HW09 frozen RNN  agrees with the rule: True (0 characters differ)
Scratch encoder  agrees with the rule: True (0 characters differ)
PyTorch encoder  agrees with the rule: True (0 characters differ)
```

### Task 06-2: Collect the Physical Evidence (0.25 pts.)

The decoded note names a locker and gives its padlock combination. The physical evidence for this case is real, and it's waiting there.

1. With your group, follow the directions in the decoded note. Your instructor will confirm the exact location, access, and collection window in class or on Canvas before the due date; **use the instructor's announcement if it differs from the note.**
2. Open the locker, find the evidence, and take a **photo of your whole group with it**. Every group member must be in the photo for credit.
3. **Leave the evidence and the lock exactly as you found them**, so other groups can complete the assignment too.
4. Add your photo to the markdown cell below so it appears in your exported HTML: in Colab, double-click the cell to edit it, then drag your image file into it (or use the image button in the editing toolbar).

If you can't access the location, contact your instructor before the due date for an alternate arrangement.

**Group photo:**

* `[PHOTO]`

## Task 07: Report the Results to Caulfield
### Task 07-1: Written Response (0.5 pts.)

1. Your scratch encoder's gradient check (Task 03-2) and its accuracy on the training phrases (Task 05-2) both came out well. What does each establish, and what does neither establish?
    * `[ANSWER]`
2. Four decoders (the rule, the RNN, and two encoders) agree on what the note says. Caulfield asks whether that proves Professor Dingler wrote it. What does the agreement establish, and what would still need to be shown by other evidence?
    * `[ANSWER]`
3. Looking back over the semester's investigation, from witness statements to the decoded note, name one place where a model's output was strong evidence and one place where it would have been easy to over-trust a model. Explain both briefly.
    * `[ANSWER]`

### *Story Progression*

The locker on the second floor of Cushing held exactly what the note promised. Caulfield bags the evidence himself; the intern is permitted to hold the flashlight and does so with great dignity.

Back at the precinct, the whole case is pinned to the board: the suspects' statements, the witness reliability scores, the damaged pages and the pixels that rebuilt them, the cut-out letters and the networks that read them, and finally the decoded confession, read four different ways. Every step was a model, and every model was checked against something independent: a library, a gradient, the exact rule, or another model built a different way.

The note tells us what someone *wanted* to say. Whether it's the truth is a question for evidence, not for a decoder, and the evidence is now in Caulfield's hands. Case closed (pending trial).

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-30124-homeworks/imgs/homework10/story_case_closed.png" class="img-responsive" alt="Evidence 4: Case closed: the whole investigation on one board, from the first witness statement to the final clue." style="display: block; max-width: 640px; width: 100%; height: auto; margin: 0 auto;"/>
    <figcaption style="text-align: center"><b>Evidence 4: Case closed: the whole investigation on one board, from the first witness statement to the final clue.</b></figcaption>
</div>

## Task 08: Export the Completed Notebook (0 pts.)

1. Complete your code and written responses, add your group photo to Task 06-2, and run the cells above in order. Compare your results with the expected and reference outputs.
2. **Save your notebook**, including its outputs. Set `GENERATE = True` below when you are ready to export.
3. In Colab, the cell captures the current notebook and its outputs. Locally, it reads the saved notebook at `NOTEBOOK_PATH`; adjust that path if you saved your copy elsewhere.
4. Download **`homework10.html`** and open it in a browser. Check that your name, NETID, code, written answers, outputs, and photo appear.

Submit only **`homework10.html`** on Canvas. The exporter does not rerun your code.

### Task 08-1: Code (0 pts.)

In [ ]:
GENERATE = False
NOTEBOOK_PATH = DATA_ROOT / 'homeworks' / 'homework10' / 'homework10.ipynb'

if GENERATE:
    import json
    import nbformat
    from nbconvert import HTMLExporter
    from pathlib import Path

    try:
        from google.colab import _message, files
    except ImportError:
        current = nbformat.read(NOTEBOOK_PATH, as_version=4)
        html_path = NOTEBOOK_PATH.with_name('homework10.html')
        in_colab = False
    else:
        captured = _message.blocking_request('get_ipynb', timeout_sec=10)
        current = nbformat.reads(json.dumps(captured['ipynb']), as_version=4)
        html_path = Path('/content/homework10.html')
        in_colab = True

    exporter = HTMLExporter()
    exporter.embed_images = True
    report_html, _ = exporter.from_notebook_node(current)
    html_path.write_text(report_html, encoding='utf-8')
    print(f'Exported {html_path.name}. Open it in a browser before submitting.')

    if in_colab:
        files.download(str(html_path))
    else:
        print(f'Saved to: {html_path}')
else:
    print('Export disabled. Set GENERATE = True when ready.')

### Task 08-1: Expected Output (0 pts.)

```text
Export disabled. Set GENERATE = True when ready.
```